# Análise de ações com Python

Este notebook acompanha as tarefas 01 a 19 do projeto em ordem, seguindo exatamente a estrutura descrita em TASKS.

- Bloco 1: coleta, inspeção e limpeza dos dados
- Bloco 2: análise estatística e medidas de risco
- Bloco 3: visualizações comparativas

In [ ]:
from pathlib import Path
import shutil
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

DATA_DIR = Path('data/raw')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# Limpeza da pasta de outputs antes de gerar os artefatos
for item in OUTPUT_DIR.iterdir():
    if item.is_dir():
        shutil.rmtree(item)
    else:
        item.unlink()
OUTPUT_DIR.mkdir(exist_ok=True)

## Bloco 1: coleta, inspeção e limpeza dos dados

### Tarefa 01 — Carregamento dos dados

**Objetivo:** Ler os arquivos CSV/XLSX do Yahoo Finance e carregar em um DataFrame pandas.

**Instruções:**
- Os arquivos estão em `data/raw/`
- Cada arquivo corresponde a uma ação (ex: `PETR4.csv`, `VALE3.csv`)
- Carregar todos os arquivos e consolidar em um único DataFrame
- A coluna de identificação da ação deve se chamar `ticker`

**Entrega esperada:** DataFrame `dados` com todos os arquivos consolidados e coluna `ticker`.


In [ ]:
## Tarefa 01 — Carregamento dos dados
# Leitura e consolidação dos arquivos CSV em um único DataFrame com a coluna ticker
def normalize_column(col: str) -> str:
    col = str(col).strip().lower()
    col = unicodedata.normalize('NFKD', col).encode('ascii', 'ignore').decode('utf-8')
    col = re.sub(r'[^a-z0-9]+', '_', col).strip('_')
    return col

def parse_numeric(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().replace('$', '').replace(',', '')
    s = s.replace('−', '-').replace('%', '')
    if s == '':
        return np.nan

    multiplier = 1.0
    if s.endswith('M'):
        multiplier = 1e6
        s = s[:-1]
    elif s.endswith('B'):
        multiplier = 1e9
        s = s[:-1]
    elif s.endswith('K'):
        multiplier = 1e3
        s = s[:-1]

    try:
        return float(s) * multiplier
    except ValueError:
        return np.nan

frames = []
for path in sorted(DATA_DIR.glob('*.csv')):
    df = pd.read_csv(path)
    if df.empty:
        continue
    df = df.rename(columns={c: normalize_column(c) for c in df.columns})
    df['ticker'] = path.stem
    frames.append(df)

dados = pd.concat(frames, ignore_index=True)
dados = dados.drop_duplicates().copy()
dados.head()


### Tarefa 02 — Inspeção inicial

**Objetivo:** Entender a estrutura dos dados antes de qualquer tratamento.

**Instruções:**
- Verificar shape (linhas e colunas)
- Listar tipos de dados de cada coluna
- Identificar valores ausentes
- Ver as primeiras e últimas linhas
- Executar o `.describe()` para uma visão geral das estatísticas básicas

**Entrega esperada:** Célula Markdown no notebook com um resumo do que foi encontrado — quantas linhas, quais colunas têm NaN, quais tipos estão incorretos.



In [ ]:
# Tarefa 02 — Inspeção inicial

print('Shape:', dados.shape)
print('Tipos de dados:')
print(dados.dtypes)
print('Valores ausentes:')
print(dados.isna().sum())
print('Primeiras linhas:')
print(dados.head())
print('Últimas linhas:')
print(dados.tail())
print('Resumo estatístico:')
print(dados.describe())

### Tarefa 03 - Limpeza dos dados

**Objetivo:** Remover ou corrigir problemas encontrados na inspeção.

**Instruções:**
- Remover linhas com valores ausentes nas colunas de preço
- Remover duplicatas
- Padronizar nomes de colunas (minúsculas, sem espaço, sem acento)
- Remover caracteres especiais de colunas numéricas (cifrão, vírgula como separador decimal)

**Entrega esperada:** DataFrame limpo, sem NaN nas colunas de preço, sem duplicatas, colunas padronizadas.


In [ ]:
# Tarefa 03 — Limpeza dos dados

for col in ['price', 'open', 'high', 'low', 'vol', 'change', 'change_pct']:
    if col in dados.columns:
        dados[col] = dados[col].apply(parse_numeric)

dados = dados.dropna(subset=['price']).drop_duplicates().copy()
dados = dados.sort_values(['ticker', 'date']).reset_index(drop=True)
dados.head()

### Tarefa 04 — Transformações

**Objetivo:** Preparar os dados para análise — tipos corretos e novas colunas úteis.

**Instruções:**
- Converter coluna de data para tipo `datetime`
- Ordenar por ticker e data
- Extrair ano e mês em colunas separadas
- Definir a data como índice

**Entrega esperada:** DataFrame com data como índice datetime, ordenado, com colunas `ano` e `mes`.

In [ ]:
# Tarefa 04 — Transformações

dados = dados.copy()
dados['date'] = pd.to_datetime(dados['date'], format='%m/%d/%Y', errors='coerce')
dados = dados.dropna(subset=['date']).copy()
dados = dados.sort_values(['ticker', 'date']).reset_index(drop=True)
dados['ano'] = dados['date'].dt.year
dados['mes'] = dados['date'].dt.month
dados = dados.set_index('date')
dados = dados.sort_values(['ticker', 'date'])
dados.head()

### Tarefa 05 — Análise estatística descritiva

**Objetivo:** Calcular as principais medidas estatísticas por ação.

**Instruções:**
- Calcular média, mediana, desvio padrão, mínimo e máximo do preço de fechamento
- Agrupar por ticker
- Apresentar em uma tabela formatada

**Entrega esperada:** DataFrame `resumo` com as métricas por ticker.

In [ ]:
# Tarefa 05 — Análise estatística descritiva

resumo = (
    dados.groupby('ticker')['price']
    .agg(['mean', 'median', 'std', 'min', 'max'])
    .round(4)
)
resumo.columns = ['media', 'mediana', 'desvio_padrao', 'minimo', 'maximo']
resumo

### Tarefa 06 — Retorno diário

**Objetivo:** Calcular o retorno percentual de um dia para o outro de cada ação.

**O que é retorno diário?**
É a variação percentual do preço de fechamento entre dois dias consecutivos.
Se o preço passou de R$ 100 para R$ 102, o retorno foi de +2%.
Preço absoluto não é comparável entre ações — retorno percentual é.

**Instruções:**
- Calcular separadamente por ticker para não misturar ações
- Remover o primeiro dia de cada ticker (gera NaN por não ter dia anterior)
- Resultado deve ficar em uma nova coluna chamada `retorno_diario`

**Entrega esperada:** Coluna `retorno_diario` no DataFrame.

In [ ]:
# Tarefa 06 — Retorno diário

dados['retorno_diario'] = dados.groupby('ticker')['price'].pct_change()
dados[['retorno_diario']].head()

### Tarefa 07 — Retorno acumulado

**Objetivo:** Mostrar quanto R$ 1,00 investido no início do período valeria ao longo do tempo em cada ação.

**O que é retorno acumulado?**
É o efeito composto dos retornos diários — multiplica os fatores de retorno dia a dia.
Permite comparar visualmente o desempenho de ações com preços absolutos muito diferentes.

**Instruções:**
- Calcular a partir do retorno diário usando produto acumulado
- Calcular separadamente por ticker
- Resultado deve ficar em uma nova coluna chamada `retorno_acumulado`

**Entrega esperada:** Coluna `retorno_acumulado` no DataFrame, partindo de 1.0 no primeiro dia de cada ticker.


In [ ]:
# Tarefa 07 — Retorno acumulado

dados['retorno_acumulado'] = dados.groupby('ticker')['retorno_diario'].transform(lambda s: (1 + s.fillna(0)).cumprod())
dados[['retorno_acumulado']].head()

### Tarefa 08 — Volatilidade anualizada

**Objetivo:** Medir o risco de cada ação de forma comparável e padronizada.

**O que é volatilidade anualizada?**
É o desvio padrão dos retornos diários multiplicado pela raiz quadrada de 252 (número de pregões no ano).
Essa conversão permite comparar a volatilidade entre ativos independentemente do período analisado.
Uma ação com volatilidade anualizada de 40% é significativamente mais arriscada que uma com 15%.

**Instruções:**
- Calcular o desvio padrão dos retornos diários por ticker
- Multiplicar por √252 para anualizar
- Apresentar em uma tabela ordenada da maior para a menor volatilidade

**Entrega esperada:** DataFrame com a volatilidade anualizada de cada ação, em ordem decrescente.


In [ ]:
# Tarefa 08 — Volatilidade anualizada

volatilidade = (
    dados.groupby('ticker')['retorno_diario']
    .std(ddof=1)
    .mul(np.sqrt(252))
    .sort_values(ascending=False)
    .to_frame('volatilidade_anualizada')
)
volatilidade

### Tarefa 09 — Drawdown

**Objetivo:** Identificar as maiores quedas de cada ação em relação ao seu pico histórico no período analisado.

**O que é drawdown?**
É a queda percentual do preço em relação ao maior valor já atingido até aquele momento.
Se uma ação chegou a R$ 100 e caiu para R$ 70, o drawdown é de -30%.
É uma medida de risco muito usada na prática porque mostra o quanto o investidor perdeu antes de a ação se recuperar.

**Instruções:**
- Calcular o pico acumulado (máximo histórico até cada data) por ticker
- Calcular o drawdown como a variação entre o preço atual e o pico
- Identificar o drawdown máximo (pior momento) de cada ação

**Entrega esperada:** Coluna `drawdown` no DataFrame e tabela com o drawdown máximo por ticker.


In [ ]:
# Tarefa 09 — Drawdown

dados['pico_preco'] = dados.groupby('ticker')['price'].cummax()
dados['drawdown'] = dados['price'] / dados['pico_preco'] - 1
drawdown_max = (
    dados.groupby('ticker')['drawdown']
    .min()
    .sort_values(ascending=True)
    .to_frame('drawdown_maximo')
)
drawdown_max

### Tarefa 10 — Índice de Sharpe

**Objetivo:** Medir a relação entre retorno e risco de cada ação.

**O que é o Índice de Sharpe?**
É o retorno médio de um ativo dividido pela sua volatilidade.
Responde à pergunta: quanto de retorno estou recebendo por cada unidade de risco que estou assumindo?
Um Sharpe de 1.0 é considerado bom. Acima de 2.0, excelente. Negativo significa que o ativo rendeu menos que a taxa livre de risco.
Neste projeto usaremos taxa livre de risco = 0 para simplificar (Sharpe não anualizado).

**Instruções:**
- Calcular o retorno médio diário por ticker
- Dividir pelo desvio padrão dos retornos diários
- Multiplicar por √252 para anualizar
- Apresentar em tabela ordenada do maior para o menor Sharpe

**Entrega esperada:** DataFrame com o Índice de Sharpe anualizado de cada ação, em ordem decrescente.


In [ ]:
# Tarefa 10 — Índice de Sharpe

retorno_medio = dados.groupby("ticker")["retorno_diario"].mean()
desvio = dados.groupby("ticker")["retorno_diario"].std(ddof=1)
retorno_medio_anualizado = (retorno_medio * np.sqrt(252)).to_frame("retorno_medio_anualizado")
sharpe = (
    (retorno_medio / desvio * np.sqrt(252))
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .sort_values(ascending=False)
    .to_frame("sharpe_anualizado")
)
retorno_medio_anualizado.to_csv(OUTPUT_DIR / "10_retorno_medio_anualizado.csv")
sharpe.to_csv(OUTPUT_DIR / "10_sharpe_anualizado.csv")

### Tarefa 11 — Correlação entre ações

**Objetivo:** Verificar se as ações se movem juntas ou de forma independente.

**O que é correlação?**
É um valor entre -1 e 1 que mede o quanto os retornos de duas ações se movem juntos.
1 = sempre na mesma direção · 0 = independentes · -1 = sempre em direções opostas.
Ações muito correlacionadas não diversificam o risco de uma carteira.

**Instruções:**
- Criar uma tabela pivô com os retornos diários de cada ação nas colunas
- Calcular a matriz de correlação entre todas as combinações de pares

**Entrega esperada:** DataFrame `correlacao` com a matriz de correlação entre as ações.


In [ ]:
# Tarefa 11 — Correlação entre ações

retornos_pivot = (
    dados.reset_index()[['date', 'ticker', 'retorno_diario']]
    .pivot(index='date', columns='ticker', values='retorno_diario')
)
correlacao = retornos_pivot.corr()
correlacao

## BLOCO 3 — Visualização

### Tarefa 12 — Gráfico de preços e médias móveis

**Objetivo:** Visualizar a evolução do preço de fechamento e identificar tendências.

**O que é média móvel?**
É a média dos últimos N dias de preço. Suaviza o ruído do dia a dia e revela a tendência de fundo.
A média de 21 dias captura tendências de curto prazo. A de 200 dias, tendências de longo prazo.
Quando o preço cruza a média de 200 dias de baixo para cima, é um sinal clássico de tendência de alta.

**Instruções:**
- Um gráfico de linha por ação com o preço de fechamento
- Adicionar as médias móveis de 21 e 200 dias no mesmo gráfico
- Título, rótulos de eixo e legenda obrigatórios

**Entrega esperada:** Gráfico de linhas com preço e médias móveis para cada ação.


In [ ]:
# Tarefa 12 — Gráfico de preços e médias móveis

for ticker in dados['ticker'].unique():
    subset = dados[dados['ticker'] == ticker].copy()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(subset.index, subset['price'], label='Preço de fechamento', linewidth=1.2)
    ax.plot(subset.index, subset['price'].rolling(21).mean(), label='Média móvel 21 dias', linestyle='--')
    ax.plot(subset.index, subset['price'].rolling(200).mean(), label='Média móvel 200 dias', linestyle=':')
    ax.set_title(f'{ticker} — Preço e médias móveis')
    ax.set_xlabel('Data')
    ax.set_ylabel('Preço')
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'12_{ticker}_precos_medias_moveis.png', dpi=200)
    plt.close(fig)

### Tarefa 13 — Gráfico de retorno acumulado

**Objetivo:** Comparar o desempenho de todas as ações no mesmo gráfico a partir de uma base comum.

**Instruções:**
- Todas as ações no mesmo gráfico
- Eixo Y começa em 1.0 (base comum de comparação)
- Linha horizontal tracejada em 1.0 como referência (abaixo = perda, acima = ganho)
- Título, rótulos de eixo e legenda obrigatórios

**Entrega esperada:** Gráfico de linhas com o retorno acumulado de todas as ações sobrepostas.


In [ ]:
# Tarefa 13 — Gráfico de retorno acumulado

fig, ax = plt.subplots(figsize=(10, 5))
for ticker, group in dados.groupby('ticker'):
    ax.plot(group.index, group['retorno_acumulado'], label=ticker)
ax.axhline(1.0, linestyle='--', color='gray', linewidth=1)
ax.set_title('Retorno acumulado das ações')
ax.set_xlabel('Data')
ax.set_ylabel('Retorno acumulado')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '13_retorno_acumulado_todas_acoes.png', dpi=200)
plt.close(fig)

### Tarefa 14 — Histograma de retornos diários

**Objetivo:** Visualizar a distribuição dos retornos de cada ação.

**Instruções:**
- Um histograma por ação com curva de densidade (KDE)
- Linha vertical tracejada no zero para referência
- Permite identificar assimetria e caudas pesadas na distribuição

**Entrega esperada:** Histogramas com curva KDE e linha de referência no zero para cada ação.


In [ ]:
# Tarefa 14 — Histograma de retornos diários

for ticker in dados['ticker'].unique():
    subset = dados[dados['ticker'] == ticker]['retorno_diario'].dropna()
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(subset, kde=True, ax=ax, bins=30)
    ax.axvline(0, linestyle='--', color='red')
    ax.set_title(f'{ticker} — Distribuição dos retornos diários')
    ax.set_xlabel('Retorno diário')
    ax.set_ylabel('Frequência')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'14_{ticker}_histograma_retornos.png', dpi=200)
    plt.close(fig)

### Tarefa 15 — Boxplot comparativo de retornos

**Objetivo:** Comparar a dispersão dos retornos entre as ações lado a lado.

**Instruções:**
- Um boxplot por ação no mesmo gráfico
- Permite identificar visualmente qual ação tem maior volatilidade e quais têm outliers extremos

**Entrega esperada:** Boxplot comparativo com todas as ações no mesmo gráfico.


In [ ]:
# Tarefa 15 — Boxplot comparativo de retornos

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=dados.reset_index(), x='ticker', y='retorno_diario', ax=ax)
ax.set_title('Boxplot comparativo de retornos diários')
ax.set_xlabel('Ticker')
ax.set_ylabel('Retorno diário')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '15_boxplot_retornos.png', dpi=200)
plt.close(fig)

### Tarefa 16 — Scatter retorno médio × volatilidade

**Objetivo:** Posicionar cada ação em um mapa de risco e retorno.

**O que este gráfico mostra?**
Cada ação vira um ponto. Eixo X é a volatilidade anualizada (risco). Eixo Y é o retorno médio anualizado.
O ideal é estar no canto superior esquerdo: alto retorno com baixo risco.
Este é o gráfico mais usado em análise de carteiras para comparar ativos.

**Instruções:**
- Um ponto por ação
- Eixo X: volatilidade anualizada · Eixo Y: retorno médio anualizado
- Identificar cada ponto com o nome do ticker
- Linhas de referência no zero em ambos os eixos

**Entrega esperada:** Scatter com um ponto por ação identificado pelo ticker.


In [ ]:
# Tarefa 16 — Scatter retorno médio × volatilidade

scatter_df = pd.concat([volatilidade, retorno_medio_anualizado], axis=1).reset_index()
scatter_df = scatter_df.rename(columns={'index': 'ticker'})
fig, ax = plt.subplots(figsize=(8, 5))
for _, row in scatter_df.iterrows():
    ax.scatter(row['volatilidade_anualizada'], row['retorno_medio_anualizado'], s=80)
    ax.text(row['volatilidade_anualizada'], row['retorno_medio_anualizado'], row['ticker'], fontsize=9)
ax.axhline(0, linestyle='--', color='gray')
ax.axvline(0, linestyle='--', color='gray')
ax.set_title('Retorno médio × Volatilidade anualizada')
ax.set_xlabel('Volatilidade anualizada')
ax.set_ylabel('Retorno médio anualizado')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '16_scatter_risco_retorno.png', dpi=200)
plt.close(fig)

### Tarefa 17 — Mapa de calor de correlação

**Objetivo:** Visualizar a matriz de correlação entre as ações de forma que facilite a identificação dos pares mais e menos correlacionados.

**Instruções:**
- Usar escala de cores divergente — vermelho para correlação negativa, azul para positiva
- Mostrar os valores numéricos dentro de cada célula
- Quanto mais intenso o azul, mais as ações se movem juntas

**Entrega esperada:** Heatmap com valores anotados e escala de cores correta.

In [ ]:
# Tarefa 17 — Mapa de calor de correlação

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(correlacao, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax)
ax.set_title('Mapa de calor da correlação entre ações')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '17_heatmap_correlacao.png', dpi=200)
plt.close(fig)

### Tarefa 18 — Scatterplot de correlação entre pares

**Objetivo:** Aprofundar a análise do par de ações com maior correlação identificado no heatmap.

**Por que usar o scatterplot após o heatmap?**
O heatmap mostra o número da correlação (ex: 0.87) mas não mostra como essa relação se comporta na prática.
O scatterplot coloca cada dia como um ponto — eixo X é o retorno de uma ação, eixo Y é o retorno da outra.
Com isso você vê se a relação é linear, se existem outliers e se a correlação se mantém em dias de alta volatilidade.

**Instruções:**
- Identificar automaticamente o par com maior correlação a partir da matriz da tarefa 11
- Cada ponto representa um dia de pregão
- Adicionar linha de tendência (regressão linear)
- Incluir o valor da correlação no título do gráfico
- Linhas de referência no zero em ambos os eixos

**Entrega esperada:** Scatterplot do par com maior correlação, com linha de tendência e valor de correlação no título.

> **Sequência analítica:** Tarefa 17 (heatmap) identifica os pares mais correlacionados → Tarefa 18 (scatterplot) aprofunda a análise do par mais relevante.


In [ ]:
# Tarefa 18 — Scatterplot de correlação entre pares

upper = correlacao.where(np.triu(np.ones(correlacao.shape), k=1).astype(bool))
pair = upper.stack().idxmax()
ticker_x, ticker_y = pair
fig, ax = plt.subplots(figsize=(8, 5))
sns.regplot(x=retornos_pivot[ticker_x], y=retornos_pivot[ticker_y], ax=ax, scatter_kws={'s': 8, 'alpha': 0.5}, line_kws={'color': 'red'})
ax.axhline(0, linestyle='--', color='gray')
ax.axvline(0, linestyle='--', color='gray')
corr_value = correlacao.loc[ticker_x, ticker_y]
ax.set_title(f'Correlação entre {ticker_x} e {ticker_y} (r = {corr_value:.2f})')
ax.set_xlabel(ticker_x)
ax.set_ylabel(ticker_y)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f'18_scatter_{ticker_x}_{ticker_y}.png', dpi=200)
plt.close(fig)

### Tarefa 19 — Análise de volume

**Objetivo:** Entender o comportamento do volume negociado e sua relação com o movimento de preços.

**O que o volume revela?**
Volume alto confirma um movimento de preço — se a ação sobe com volume alto, a tendência é mais confiável.
Volume baixo em uma alta pode indicar movimento fraco, sem convicção dos investidores.
É um dos indicadores mais usados para validar sinais de compra e venda.

**Instruções:**
- Gráfico de barras do volume diário por ação
- Calcular o volume médio diário por ticker e apresentar em tabela
- Identificar os 5 dias de maior volume de cada ação e verificar se coincidiram com grandes variações de preço

**Entrega esperada:** Gráfico de volume, tabela de volume médio e tabela dos 5 dias de maior volume com a variação de preço correspondente.


In [ ]:
# Tarefa 19 — Análise de volume

for ticker in dados['ticker'].unique():
    subset = dados[dados['ticker'] == ticker].copy()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(
        subset.index,
        subset['vol'],
        color='#1f77b4',
        edgecolor='black',
        linewidth=0.8,
        alpha=0.9,
    )
    ax.set_title(f'{ticker} — Volume diário')
    ax.set_xlabel('Data')
    ax.set_ylabel('Volume')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'19_{ticker}_volume_diario.png', dpi=200)
    plt.close(fig)

volume_medio = dados.groupby('ticker')['vol'].mean().sort_values(ascending=False).to_frame('volume_medio_diario')
volume_medio

In [ ]:
# Tarefa 19 — Top 5 dias de maior volume

top_volume = (
    dados.reset_index()[['date', 'ticker', 'vol', 'price']]
    .sort_values(['ticker', 'vol'], ascending=[True, False])
    .groupby('ticker')
    .head(5)
    .copy()
)
top_volume['variacao_preco'] = top_volume.groupby('ticker')['price'].pct_change()
top_volume